In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from Utilities_copy import extractor
import uproot
import awkward as ak    

x_MH200=extractor("/home/riccardo/Tesi/Cartella_Analisi_Dati/Dati/Tprime_tAq_1800_MH200_LH_2017.root", "Events")


file=uproot.open("/home/riccardo/Tesi/Cartella_Analisi_Dati/Dati/Tprime_tAq_1800_MH200_LH_2017.root")
tree=file["Events"]
booleans= tree.arrays(["FatJet_isMatchedWithA"], library="ak")
booleanas=tree.arrays(["FatJet_isMatchedWith2BHadrons"], library="ak")
Fatjet_isMatchedWithA= booleans["FatJet_isMatchedWithA"]
Fatjet_isMatchedWith2BHadrons= booleanas["FatJet_isMatchedWith2BHadrons"]
#Filtriamo i dati

mask = (ak.flatten(Fatjet_isMatchedWithA) == 1) & (ak.flatten(Fatjet_isMatchedWith2BHadrons) == 1)
x_filtered = x_MH200[mask]

from scipy.special import voigt_profile
from iminuit import Minuit
from iminuit.cost import LeastSquares

x_plot=list(x_filtered)
x_plot.sort()
x_easy=[x for x in x_plot if 150 < x < 250]

def voigt2(x, norm, mu, sigma, gamma, norm2, mu2, sigma2, gamma2):
    return voigt_profile(x-mu, sigma, gamma) * norm + norm2*voigt_profile(x-mu2, sigma2, gamma2)

bin_counts, bin_edges = np.histogram(x_easy, bins=50)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
bin_width = bin_edges[1] - bin_edges[0]
bin_densities = bin_counts / (len(x_easy) * bin_width)  # Densità normalizzata
yerr=np.sqrt(bin_counts) / (len(x_easy) * bin_width) # Errore standard per i dati binned

ls_voigt=LeastSquares(bin_centers, bin_densities, yerr, model=voigt2)

m_voigt=Minuit(ls_voigt,  norm=1, mu=200, sigma=5, gamma=1, norm2=1, mu2=200, sigma2=5, gamma2=0.001)
m_voigt.limits["mu"]= (150, 250)
m_voigt.limits["sigma"]= (0.1, 20)
m_voigt.limits["gamma"]= (0.01, 10)
m_voigt.fixed["gamma2"]= True

m_voigt.migrad()

/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/cppyy/__init__.py:374: UserWarning: CPyCppyy API not found (tried: /home/riccardo/anaconda3/envs/rootnev/include/site/python3.14); set CPPYY_API_PATH envar to the 'CPyCppyy' API directory to fix
  warnings.warn("CPyCppyy API not found (tried: %s); "
/home/riccardo/anaconda3/envs/rootnev/lib/python3.14/site-packages/awkward/_nplikes/array_module.py:289: RuntimeWarning: invalid value encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 90.34 (χ²/ndof = 2.1)      │              Nfcn = 835              │
│ EDM = 0.000117 (Goal: 0.0002)    │            time = 0.3 sec            │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬────────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name   │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼────────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ norm   │   0.66    │   0.04    │            │            │         │         │       │
│ 1 │ mu     │  206.16   │   0.19    │            │            │   150   │   250   │       │
│ 2 │ sigma  │   11.06   │   0.21    │            │            │   0.1   │   20    │       │
│ 3 │ gamma  │    1.2    │    0.4    │            │            │  0.01   │   10    │       │
│ 4 │ norm2  │   0.36    │   0.04    │            │            │         │         │       │
│ 5 │ mu2    │   185.9   │    2.0    │            │            │         │         │       │
│ 6 │ sigma2 │   20.4    │    0.9    │            │            │         │         │       │
│ 7 │ gamma2 │  1.00e-3  │  0.01e-3  │            │            │         │         │  yes  │
└───┴────────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌────────┬─────────────────────────────────────────────────────────────────┐
│        │    norm      mu   sigma   gamma   norm2     mu2  sigma2  gamma2 │
├────────┼─────────────────────────────────────────────────────────────────┤
│   norm │ 0.00193 -0.0037  0.0029  0.0130 -0.0018 -0.0871 -0.0332  0.0000 │
│     mu │ -0.0037  0.0357   -0.02   -0.01  0.0035    0.16    0.00    0.00 │
│  sigma │  0.0029   -0.02  0.0432   -0.02 -0.0029   -0.15   -0.01    0.00 │
│  gamma │  0.0130   -0.01   -0.02   0.133 -0.0120   -0.56   -0.27    0.00 │
│  norm2 │ -0.0018  0.0035 -0.0029 -0.0120 0.00171  0.0822  0.0314  0.0000 │
│    mu2 │ -0.0871    0.16   -0.15   -0.56  0.0822    4.06     1.5       0 │
│ sigma2 │ -0.0332    0.00   -0.01   -0.27  0.0314     1.5   0.785     0.0 │
│ gamma2 │  0.0000    0.00    0.00    0.00  0.0000       0     0.0       0 │
└────────┴─────────────────────────────────────────────────────────────────┘

In [ ]:
fit_MH200_values={}
fit_MH200_errors={}

fit_values={'MH200': fit_MH200_values,}
fit_errors={'MH200_errors': fit_MH200_errors}



for param in m_voigt.parameters:
    fit_MH200_values[param] = m_voigt.values[param]

for error in m_voigt.parameters:    #Qui non ho capito come fa a capire che deveestarre gli errori 
    fit_MH200_errors[error] = m_voigt.errors[error]

print(fit_MH200_values)
print(fit_MH200_errors)

import json
#QUi sono andato di metodo oragutang, ho deciso di voler fare 2 file separati peer errori e valori 
#Ho tenuto lo stesso quello con tutti i valori, casomai cambiassi idea

with open("fit_results.json", "r") as f:
    results=json.load(f)

with open("fit_values.json", "r") as g:
    values=json.load(g) 

with open("fit_errors.json", "r") as h:
    errors=json.load(h)


results["MH200"]=fit_MH200_values
results["MH200_errors"]=fit_MH200_errors

with open("fit_results.json", "w") as f:
    json.dump(results, f, indent=1)

values["MH200"]=fit_MH200_values
with open("fit_values.json", "w") as g:
    json.dump(values, g, indent=1)  

errors["MH200_errors"]=fit_MH200_errors
with open("fit_errors.json", "w") as h:
    json.dump(errors, h, indent=1)  

{'norm': 0.6598718712321834, 'mu': 206.15757663467727, 'sigma': 11.060971290392189, 'gamma': 1.155194506556951, 'norm2': 0.3631236498233513, 'mu2': 185.88177352240427, 'sigma2': 20.43678384178792, 'gamma2': 0.001}
{'norm': 0.04388675259964073, 'mu': 0.1889831232922461, 'sigma': 0.20785719468824482, 'gamma': 0.3639315022233919, 'norm2': 0.04131141034139166, 'mu2': 2.015982913305664, 'sigma2': 0.8860612922558208, 'gamma2': 1e-05}
